# 1.02 PyTorch 张量（Tensor）

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/01_prerequisites/1.02_pytorch_tensor.ipynb)


## 为什么要学

张量是 PyTorch 里的"数组"，和 NumPy 数组几乎一样，多了两个能力：**能放到 GPU 上算**、**能自动求导**（下一课讲）。VLA 模型的每一行代码都在做张量变形：把图像切成 patch、把图像 token 和文字 token 拼起来、把动作块展平……

本课的目标：**看到任何一行张量操作，都能说出输入输出的形状**。

---

## 1. 创建张量

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import torch
import numpy as np

a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])     # 从 Python 列表创建
b = torch.zeros(2, 3)                           # 全 0
c = torch.ones(2, 3, dtype=torch.long)          # 指定类型
d = torch.randn(2, 3)                           # 标准正态分布随机数
e = torch.arange(6)                             # 0..5
f = torch.linspace(0, 1, 5)                     # 等间隔
g = torch.zeros_like(d)                         # 和 d 形状、类型都相同的全 0 张量
print(a.shape, a.dtype, b.dtype, c.dtype, e)

# 与 NumPy 互转（CPU 上共享内存，改一个另一个也变）
n = np.arange(3.0)
t = torch.from_numpy(n)
n[0] = 100
print(t)                # tensor([100., 1., 2.])
print(t.numpy())

### dtype：类型很重要

| dtype | 用途 |
|---|---|
| `torch.float32` | 默认浮点，训练常用 |
| `torch.bfloat16` | 大模型训练/推理（π0 用它节省显存），精度低但范围大 |
| `torch.long` (int64) | 索引、token id、类别标签 |
| `torch.bool` | 掩码（mask） |

In [ ]:
x = torch.tensor([1, 2, 3])
print(x.dtype)                 # int64
print(x.float().dtype)         # float32
print(x.to(torch.bfloat16))
# 常见错误：整数张量做除法、或者把 float 当作索引

### device：CPU 还是 GPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.randn(3, 3).to(device)
y = torch.randn(3, 3, device=device)  # 直接在目标设备上创建
print((x @ y).device)
# 规则：参与同一个运算的张量必须在同一个 device 上，否则报错

---

## 2. 变形：view / reshape / permute / transpose

In [ ]:
x = torch.arange(24)            # (24,)
y = x.view(2, 3, 4)             # (2, 3, 4)：只改变"看待方式"，不复制数据
z = x.reshape(4, -1)            # (4, 6)：-1 表示"自动推算"
print(y.shape, z.shape)

img = torch.randn(8, 224, 224, 3)    # 很多数据集存成 (B, H, W, C)
img_chw = img.permute(0, 3, 1, 2)    # PyTorch 卷积需要 (B, C, H, W)
print(img_chw.shape)

q = torch.randn(2, 10, 4, 16)        # (B, L, heads, head_dim)
q_t = q.transpose(1, 2)              # (B, heads, L, head_dim)：交换两个维度
print(q_t.shape)

> **`view` vs `reshape`**：`permute/transpose` 之后张量在内存中不再连续，`view` 会报错，要用 `reshape`（或先 `.contiguous()`）。不确定时用 `reshape`。

### 增删维度：unsqueeze / squeeze / None

In [ ]:
s = torch.randn(4, 7)         # (B, D) 一批状态
print(s.unsqueeze(1).shape)   # (4, 1, 7)：在第 1 维插入长度为 1 的维度
print(s[:, None, :].shape)    # 同上，更常用的写法
print(torch.randn(4, 1, 7).squeeze(1).shape)   # (4, 7)：删掉长度为 1 的维度

### 拼接：cat vs stack

In [ ]:
img_tokens = torch.randn(2, 256, 64)   # (B, 256 个图像 token, D)
txt_tokens = torch.randn(2, 48, 64)    # (B, 48 个文字 token, D)
prefix = torch.cat([img_tokens, txt_tokens], dim=1)   # 沿已有维度拼接 -> (2, 304, 64)
print(prefix.shape)                                   # 这就是 π0 里 embed_prefix 做的事

frames = [torch.randn(3, 64, 64) for _ in range(5)]
video = torch.stack(frames, dim=0)     # 新建一个维度堆叠 -> (5, 3, 64, 64)
print(video.shape)

### 扩展：expand vs repeat

In [ ]:
token = torch.randn(1, 1, 64)          # 一个可学习的特殊 token（比如 ViT 的 CLS、RLT 的 RL token）
B = 8
print(token.expand(B, -1, -1).shape)   # (8, 1, 64)：不复制内存，只是"看起来"复制了
print(token.repeat(B, 1, 1).shape)     # (8, 1, 64)：真的复制内存

---

## 3. 索引进阶：gather 与布尔掩码

强化学习里常要"取出每个样本所选动作对应的值"：Q 值表 `(B, A)`，动作 `(B,)`，要得到 `(B,)`。

In [ ]:
q_values = torch.tensor([[1.0, 2.0, 3.0],
                         [4.0, 5.0, 6.0]])     # (B=2, A=3)
actions = torch.tensor([2, 0])                 # 第 0 个样本选动作 2，第 1 个选动作 0
picked = q_values.gather(1, actions[:, None]).squeeze(1)
print(picked)          # tensor([3., 4.])
# 等价写法：高级索引
print(q_values[torch.arange(2), actions])

In [ ]:
# 布尔掩码：把 padding 位置的 loss 置零（序列模型里天天用）
loss = torch.rand(2, 5)                          # (B, T)
mask = torch.tensor([[1, 1, 1, 0, 0],
                     [1, 1, 1, 1, 1]], dtype=torch.bool)
masked_mean = (loss * mask).sum() / mask.sum()   # 只对有效位置求平均
print(masked_mean)

---

## 4. einsum：用"爱因斯坦求和"描述任意张量乘法

`einsum` 用字母标记每一维，出现在输入但不出现在输出的字母会被**求和**。它是读写注意力代码的利器。

In [ ]:
A = torch.randn(3, 4); B_ = torch.randn(4, 5)
print(torch.allclose(torch.einsum("ik,kj->ij", A, B_), A @ B_))   # 矩阵乘法

x = torch.randn(2, 10, 64); W = torch.randn(64, 32)
print(torch.einsum("bld,de->ble", x, W).shape)                   # 对最后一维做线性变换

q = torch.randn(2, 4, 10, 16); k = torch.randn(2, 4, 12, 16)      # (B, heads, L, d)
scores = torch.einsum("bhqd,bhkd->bhqk", q, k)                    # 注意力分数 (B, heads, Lq, Lk)
print(scores.shape)

---

## 练习

### 练习 1：把图像切成 patch（ViT 的第一步）

把图像 `(B, C, H, W)` 切成不重叠的 `P×P` 小块，每块展平成一个向量，输出 `(B, N, P*P*C)`，其中 `N = (H/P)·(W/P)`。patch 的顺序是**从左到右、从上到下**。每个 patch 内部的展平顺序为 `(P, P, C)`。

> SigLIP（π0 的视觉编码器）把 224×224 的图切成 14×14 的 patch，得到 16×16=256 个图像 token。

In [ ]:
def patchify(images, P):
    """images: (B, C, H, W) -> (B, N, P*P*C)"""
    B, C, H, W = images.shape
    # TODO: 提示：reshape 成 (B, C, H/P, P, W/P, P)，再 permute 成 (B, H/P, W/P, P, P, C)，最后 reshape
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def patchify(images, P):
    """images: (B, C, H, W) -> (B, N, P*P*C)"""
    B, C, H, W = images.shape
    assert H % P == 0 and W % P == 0, "图像边长必须能被 patch 大小整除"
    x = images.reshape(B, C, H // P, P, W // P, P)   # 把 H 拆成 (行块号, 块内行)，W 同理
    x = x.permute(0, 2, 4, 3, 5, 1)                  # (B, H/P, W/P, P, P, C)
    return x.reshape(B, (H // P) * (W // P), P * P * C)

In [ ]:
# ✅ 自测
imgs = torch.randn(2, 3, 8, 8)
p = patchify(imgs, 4)
assert p.shape == (2, 4, 48)
# 第 1 个 patch（右上角）应等于 imgs[:, :, 0:4, 4:8] 按 (P, P, C) 展平
want = imgs[:, :, 0:4, 4:8].permute(0, 2, 3, 1).reshape(2, -1)
assert torch.allclose(p[:, 1], want)
print("通过 ✅")

### 练习 2：用 einsum 实现批量注意力加权

给定注意力权重 `attn (B, H, Lq, Lk)` 和值 `v (B, H, Lk, d)`，计算输出 `(B, H, Lq, d)`：`out[b,h,i] = Σ_j attn[b,h,i,j] · v[b,h,j]`。

In [ ]:
def attn_apply(attn, v):
    # TODO: 一行 einsum
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def attn_apply(attn, v):
    # j（即 k 维）同时出现在两个输入中、不出现在输出中 -> 被求和
    return torch.einsum("bhqk,bhkd->bhqd", attn, v)

In [ ]:
# ✅ 自测
attn = torch.softmax(torch.randn(2, 4, 5, 6), dim=-1); v = torch.randn(2, 4, 6, 8)
assert torch.allclose(attn_apply(attn, v), attn @ v, atol=1e-6)
print("通过 ✅（矩阵乘法 @ 对最后两维做乘法、前面的维度自动当作 batch）")

### 练习 3：取出所选动作的对数概率（策略梯度要用）

策略网络输出每个离散动作的对数概率 `logp (B, T, A)`，实际执行的动作 `actions (B, T)`（整数）。返回 `(B, T)`：每一步所选动作的对数概率。

In [ ]:
def select_logp(logp, actions):
    # TODO: 用 gather，注意 index 需要和 logp 维数相同
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def select_logp(logp, actions):
    # actions[..., None]: (B, T, 1)，在最后一维(动作维)上 gather，得到 (B, T, 1)，再去掉最后一维
    return logp.gather(-1, actions[..., None]).squeeze(-1)

In [ ]:
# ✅ 自测
logp = torch.log_softmax(torch.randn(3, 4, 5), dim=-1)
acts = torch.randint(0, 5, (3, 4))
out = select_logp(logp, acts)
assert out.shape == (3, 4)
assert torch.isclose(out[2, 1], logp[2, 1, acts[2, 1]])
print("通过 ✅")

---

## 小结

| 操作 | 记忆要点 |
|---|---|
| `reshape/view` | 总元素数不变；`-1` 自动推算 |
| `permute/transpose` | 交换维度顺序（数据排列改变） |
| `unsqueeze / None` | 插入长度 1 的维度，配合广播 |
| `cat` vs `stack` | 沿已有维度拼 vs 新建维度堆 |
| `expand` vs `repeat` | 不复制 vs 复制内存 |
| `gather` | 按索引在某一维上"挑元素" |
| `einsum` | 字母标维度，消失的字母被求和 |

**思考题**：`patchify` 里如果把 `permute(0, 2, 4, 3, 5, 1)` 写成 `permute(0, 2, 4, 5, 3, 1)`，自测还能通过吗？如果模型从零训练，这个 bug 影响大吗？如果加载的是**预训练好的** SigLIP 权重呢？（提示：从零训练时线性层可以"学会"任何固定的像素排列；预训练权重却只认识原来的排列。这类"不报错的 bug"是迁移预训练模型时最难查的。）